<h1 style="color:#F0B4B4; font-size:32px;">Import modules</h1>

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import json
from pathlib import Path


<h1 style="color:#F0B4B4; font-size:32px;">Loading datasets</h1>

In [ ]:
USA_SWIMMING_DIR = Path(
    "/Users/nouhailaimaneabbassi/Desktop/Pacing/app/data/cleaned_data/usaswimming"
)

rows: list[dict] = []
file_count = 0
MAX_FILES = None  

# Chaque fichier contient une liste de performances (dicts). On aplatit tout.
for i, file in enumerate(USA_SWIMMING_DIR.rglob("*.json")):
    with open(file, "r", encoding="utf-8") as f:
        if MAX_FILES is not None and i >= MAX_FILES:
            break
        payload = json.load(f)

    if isinstance(payload, list):
        rows.extend([rec for rec in payload if isinstance(rec, dict)])
    elif isinstance(payload, dict):
        rows.append(payload)

    file_count += 1

df = pd.DataFrame(rows)
print("Nombre de fichiers:", file_count)
print("Nombre de performances:", df.shape[0])
print("Colonnes:", list(df.columns))


In [ ]:
# Typage/normalisation minimale pour les graphiques.
for col in ["SwimTimeSeconds", "Speed", "Distance", "PoolLength"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if "SwimDate" in df.columns:
    df["SwimDate"] = pd.to_datetime(df["SwimDate"], errors="coerce")

print("Nombre de SwimTimeSeconds non nuls:", df["SwimTimeSeconds"].notna().sum() if "SwimTimeSeconds" in df.columns else 0)
print("Nombre de Speed non nuls:", df["Speed"].notna().sum() if "Speed" in df.columns else 0)


<h1 style="color:#F0B4B4; font-size:32px;">Histogram of swim times</h1>

In [ ]:
from matplotlib.ticker import MaxNLocator

nom_event = "50 FR LCM"

swim_times = df.loc[
    (df["Event"] == nom_event)
    & (df["SwimTimeSeconds"].notna())
    & (df["SwimTimeSeconds"] < 500),
    "SwimTimeSeconds",
].dropna()

print(f"Nombre de performances pour l'event {nom_event}: {len(swim_times)}")

if len(swim_times) == 0:
    print("Aucune donnee pour cet evenement. Changez `nom_event`.")
else:
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.hist(
        swim_times,
        bins=50,
        color="#004080",
        edgecolor="#004080",
        alpha=0.7,
    )

    mean_time = float(np.mean(swim_times))
    median_time = float(np.median(swim_times))

    ax.axvline(
        mean_time,
        color="red",
        linestyle="dashed",
        linewidth=2,
        label=f"Moyenne: {mean_time:.2f}s",
    )
    ax.axvline(
        median_time,
        color="orange",
        linestyle="dashed",
        linewidth=2,
        label=f"Mediane: {median_time:.2f}s",
    )

    ax.set_title(
        f"Distribution des temps de nage pour {nom_event} (temps < 500 s)",
        fontsize=14,
    )
    ax.set_xlabel("Temps (secondes)")
    ax.set_ylabel("Nombre de performances")
    ax.legend()

    ax.grid(axis="y", alpha=0.3)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=10))
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

    plt.show()


<h1 style="color:#F0B4B4; font-size:32px;">Histogram with Kernel Density Estimation</h1>

In [ ]:
nom_event = "400 MD LCM"

swim_times = df.loc[
    (df["Event"] == nom_event)
    & (df["SwimTimeSeconds"].notna())
    & (df["SwimTimeSeconds"] < 500),
    "SwimTimeSeconds",
]

if len(swim_times) == 0:
    print("Aucune donnee pour cet evenement. Changez `nom_event`.")
else:
    plt.figure(figsize=(10, 6))
    sns.histplot(
        swim_times,
        bins=30,
        kde=True,
        color="#004080",
        edgecolor="#004080",
        alpha=0.6,
    )
    plt.title(f"Distribution des temps de natation pour {nom_event} avec densite")
    plt.xlabel("Temps (secondes)")
    plt.ylabel("Nombre de performances")
    plt.xticks(np.arange(0, 501, 25), rotation=45)
    plt.grid(axis="x", alpha=0.3)
    plt.show()


<h1 style="color:#F0B4B4; font-size:32px;">Cumulative Histogram</h1>

In [ ]:
nom_event = "200 MD LCM"

swim_times = df.loc[
    (df["Event"] == nom_event)
    & (df["SwimTimeSeconds"].notna())
    & (df["SwimTimeSeconds"] < 500),
    "SwimTimeSeconds",
]

if len(swim_times) == 0:
    print("Aucune donnee pour cet evenement. Changez `nom_event`.")
else:
    plt.figure(figsize=(10, 6))
    plt.hist(
        swim_times,
        bins=30,
        cumulative=True,
        color="#008080",
        edgecolor="black",
        alpha=0.7,
    )
    plt.title(f"Histogramme cumulatif des temps de natation pour {nom_event}")
    plt.xlabel("Temps (secondes)")
    plt.ylabel("Nombre cumule de performances")
    plt.xticks(np.arange(0, 501, 25), rotation=45)
    plt.grid(axis="x", alpha=0.3)
    plt.show()


<h1 style="color:#F0B4B4; font-size:32px;">Countplot number of performances by gender (global)</h1>

In [ ]:
df_gender = df[df["Gender"].notna()].copy()

plt.figure(figsize=(6, 4))
palette_colors = {
    "F": "#F585BD",
    "M": "#4FA2F6",
}
sns.countplot(x="Gender", data=df_gender, palette=palette_colors)
plt.title("Nombre de performances par sexe")
plt.xlabel("Sexe")
plt.ylabel("Nombre de performances")
plt.show()


<h1 style="color:#F0B4B4; font-size:32px;">Barplot of average time per swimming stroke and distance</h1>

In [ ]:
distance_choisie = 200

subset = df[
    (df["SwimTimeSeconds"].notna())
    & (df["Distance"].notna())
    & (df["Distance"] == distance_choisie)
]

if len(subset) == 0:
    print(f"Aucune donnee disponible pour la distance {distance_choisie} m")
else:
    plt.figure(figsize=(8, 5))
    sns.barplot(
        x="Stroke",
        y="SwimTimeSeconds",
        data=subset,
        estimator=np.mean,
        errorbar=None,
        palette="Set2",
    )
    plt.title(f"Temps moyen par type de nage pour la distance {distance_choisie} m")
    plt.xlabel("Type de nage")
    plt.ylabel("Temps moyen (secondes)")
    plt.ylim(0, float(subset["SwimTimeSeconds"].max()) + 5)
    plt.show()


<h1 style="color:#F0B4B4; font-size:32px;">Bar Chart: Number of Performances per Meet (Top 10)</h1>

In [ ]:
if "Meet" not in df.columns:
    print("La colonne `Meet` n'existe pas dans le dataset.")
else:
    top_meets = df["Meet"].value_counts().nlargest(10)
    plt.figure(figsize=(12, 6))
    sns.barplot(x=top_meets.index, y=top_meets.values, color="#8C5CE4")
    plt.title("Top 10 des meets par nombre de participations")
    plt.xlabel("Meet")
    plt.ylabel("Nombre de participations")
    plt.xticks(rotation=90)
    plt.show()


<h1 style="color:#F0B4B4; font-size:32px;">Line Plot of Swim Times by Stroke Type Over Time for a Sample</h1>

In [ ]:
df_plot = df[df["SwimDate"].notna() & df["SwimTimeSeconds"].notna()].copy()

if len(df_plot) == 0:
    print("Aucune donnee valide (SwimDate/SwimTimeSeconds).")
else:
    n = min(5000, len(df_plot))
    sample = df_plot.sample(n=n, random_state=0)
    plt.figure(figsize=(14, 6))
    sns.lineplot(
        x="SwimDate",
        y="SwimTimeSeconds",
        data=sample,
        hue="Stroke",
        alpha=0.7,
        estimator=np.mean,
        errorbar=None,
    )
    plt.title("Evolution des temps de nage dans le temps (echantillon)")
    plt.xlabel("Date")
    plt.ylabel("Temps de nage (s)")
    plt.legend(title="Stroke")
    plt.show()


<h1 style="color:#F0B4B4; font-size:32px;">Heatmap of Average Swim Times by Distance and Stroke Type</h1>

In [ ]:
pivot = df.pivot_table(
    values="SwimTimeSeconds",
    index="Distance",
    columns="Stroke",
    aggfunc="mean",
)

plt.figure(figsize=(12, 6))
sns.heatmap(pivot.sort_index(), annot=True, fmt=".1f", cmap="coolwarm_r")
plt.title("Moyenne des temps (en secondes) par distance et type de nage")
plt.xlabel("Type de nage")
plt.ylabel("Distance (m)")
plt.show()


<h1 style="color:#F0B4B4; font-size:32px;">Lineplot of Speed Performance Across Distances by Stroke</h1>

In [ ]:
df_speed = df[df["Speed"].notna() & df["Distance"].notna()].copy()

if len(df_speed) == 0:
    print("Aucune donnee de vitesse disponible.")
else:
    plt.figure(figsize=(10, 6))
    sns.lineplot(
        x="Distance",
        y="Speed",
        hue="Stroke",
        data=df_speed,
        errorbar=None,
        estimator=np.mean,
    )

    max_distance = int(df_speed["Distance"].max())
    plt.xticks(np.arange(0, max_distance + 50, 50), rotation=45)
    plt.title("Swimming Speed by Distance and Stroke Type")
    plt.xlabel("Distance (m)")
    plt.ylabel("Speed (m/s)")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.show()


<h1 style="color:#F0B4B4; font-size:32px;">Heatmap of Average Swim Speed by Distance and Stroke</h1>

In [ ]:
pivot = df.pivot_table(
    values="Speed",
    index="Distance",
    columns="Stroke",
    aggfunc="mean",
)

plt.figure(figsize=(10, 6))
sns.heatmap(pivot.sort_index(), annot=True, cmap="coolwarm_r", fmt=".2f")
plt.title("Vitesse moyenne selon distance et type de nage")
plt.xlabel("Stroke")
plt.ylabel("Distance (m)")
plt.show()


<div style="color:#888; font-size:14px;">
Remarque: les graphiques dependant de `splits` ne sont pas inclus ici, car le dataset cleaned `usaswimming` ne contient pas `splits`.
</div>
